# Figure 5D

In [ ]:
library(DSS)
library(bsseq)
library(data.table)
library(BiocParallel)
library(GenomicRanges)
library(scales)

In [ ]:
project_dir <- getwd()  

data_dir <- file.path(project_dir, "GSE63123_RAW")

sample_info <- data.frame(
  sample_id = c("Normal1", "Normal2", "Normal3",
                "Tumor1", "Tumor2", "Tumor3", "Tumor4", "Tumor5", "Tumor6", "Tumor7"),
  group = c(rep("Normal", 3), rep("Tumor", 7)),
  file = c("N1.txt", "N2.txt", "N3.txt",
           "T1.txt", "T2.txt", "T3.txt", "T4.txt", "T5.txt", "T6.txt", "T7.txt"),
  stringsAsFactors = FALSE
)

sample_info$path <- file.path(data_dir, sample_info$file)
print(sample_info)

stopifnot(all(file.exists(sample_info$path)))

In [ ]:
bs_list <- lapply(sample_info$path, function(p) fread(p, header = TRUE))
BSobj <- makeBSseqData(bs_list, sampleNames = sample_info$sample_id)

pData(BSobj)$group <- sample_info$group
pData(BSobj)$col <- ifelse(pData(BSobj)$group == "Normal", "#2166AC", "#B2182B")

group_vector <- sample_info$group
BSobj_collapsed <- collapseBSseq(BSobj, group = group_vector)

pData(BSobj_collapsed)$col <- ifelse(
  colnames(BSobj_collapsed) == "Normal", "#2166AC", "#B2182B"
)

BSobj_collapsed

In [ ]:
BSobj_smoothed <- BSmooth(
  BSseq = BSobj_collapsed,
  ns = 70,
  h = 500,
  maxGap = 1e6,
  BPPARAM = MulticoreParam(workers = 20),
  verbose = TRUE
)

coverage_mat <- getCoverage(BSobj_smoothed)
qc_summary <- list(
  sample_names = colnames(BSobj_smoothed),
  mean_coverage_per_group = round(colMeans(coverage_mat), 2),
  num_cpg_total = length(BSobj_smoothed),
  num_cpg_all_zero = sum(rowSums(coverage_mat) == 0)
)
qc_summary

In [ ]:
dml_test <- DMLtest(
  BSobj_collapsed,
  group1 = "Tumor",
  group2 = "Normal",
  smoothing = TRUE
)
dmls <- callDML(dml_test, p.threshold = 1e-3)
dmrs <- callDMR(dml_test, p.threshold = 1e-3, delta = 0.10)
head(dmrs)

In [ ]:
if (nrow(dmrs) > 0) {
  region1 <- GRanges(
    seqnames = dmrs$chr[1],
    ranges = IRanges(start = dmrs$start[1], end = dmrs$end[1])
  )
  pdf("DSS_first_DMR_plot.pdf", width = 8, height = 5)
  plotRegion(
    BSseq = BSobj_smoothed,
    region = region1,
    extend = 2000,
    main = "Top DSS DMR",
    addPoints = TRUE
  )
  dev.off()
}

fbxw2_region <- GRanges(
  seqnames = "chr9",
  ranges = IRanges(start = 120751978, end = 120793438)
)
pdf("FBXW2_region_plot.pdf", width = 10, height = 5)
plotRegion(
  BSseq = BSobj_smoothed,
  region = fbxw2_region,
  extend = 0,
  addRegions = if (exists("dmrs_anno_gr")) dmrs_anno_gr else NULL,
  regionCol = alpha("blue", 0.20),
  main = "FBXW2 Region with DMR Highlights",
  addTicks = FALSE,
  addPoints = TRUE
)
dev.off()

In [2]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 20.04.6 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/liblapack.so.3;  LAPACK version 3.9.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C               LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8    LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C             LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Asia/Shanghai
tzcode source: system (glibc)

attached base packages:
[1] parallel  stats4    stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] scales_1.4.0                data.table_1.16.0           DSS_2.52.0                  bsseq_1.40.0               
 [5] SummarizedExperiment_1.34.0 MatrixGenerics_1.16.0       